In [2]:
import os
import shutil
import pandas as pd

# Set iteration of study and source path
study = 'Study5.0'
condition = 'Accommodate'
source_dir = "/Users/sm6511/Downloads/acc_9/"

dest_dir = (
    f"/Users/sm6511/Desktop/Prediction-Accomodation-Exp/"
    f"data/{study}/{condition}"
)

target_dates = [
    "2026-06-10"
]

DELETE_FAILED_ATTENTION = True

completion_col = "button_end.numClicks"

if condition.lower() == "predict":
    attention_col = "answer_3_right.numClicks"
elif condition.lower() == "accommodate":
    attention_col = "button_3_correct.numClicks"
else:
    raise ValueError("condition must be 'Predict' or 'Accommodate'")

os.makedirs(dest_dir, exist_ok=True)

moved = []
skipped = []
deleted_failed_attention = []

# collect files sorted by earlier date
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 1: delete failed attention checks
if DELETE_FAILED_ATTENTION:
    for fname in candidate_files:
        src_path = os.path.join(source_dir, fname)

        try:
            df = pd.read_csv(src_path)
            df.columns = df.columns.str.strip()
        except Exception as e:
            print(f"Could not read {fname}: {e}")
            skipped.append((fname, "read error during attention check"))
            continue

        if attention_col not in df.columns:
            print(f"Missing {attention_col} in {fname}")
            skipped.append((fname, "missing attention column"))
            continue

        passed_attention = (
            pd.to_numeric(df[attention_col], errors="coerce")
            .eq(1)
            .any()
        )

        if not passed_attention:
            print(f"🗑️ DELETING FAILED ATTENTION CHECK: {fname}")
            os.remove(src_path)
            deleted_failed_attention.append(fname)

# refresh list of files
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 2: move completed files, earliest first
for fname in candidate_files:
    src_path = os.path.join(source_dir, fname)

    try:
        df = pd.read_csv(src_path)
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"Could not read {fname}: {e}")
        skipped.append((fname, "read error"))
        continue

    if completion_col not in df.columns:
        print(f"Missing {completion_col} in {fname}")
        skipped.append((fname, "missing completion column"))
        continue

    is_complete = (
        pd.to_numeric(df[completion_col], errors="coerce")
        .eq(1)
        .any()
    )

    if is_complete:
        dest_path = os.path.join(dest_dir, fname)
        shutil.move(src_path, dest_path)
        moved.append(fname)
        print(f"MOVED: {fname}")
    else:
        skipped.append((fname, f"{completion_col} != 1"))

print("\n===== SUMMARY =====")

print(f"Deleted failed attention files ({len(deleted_failed_attention)}):")
for f in deleted_failed_attention:
    print(f"  {f}")

print(f"\nMoved files ({len(moved)}):")
for f in moved:
    print(f"  {f}")

print(f"\nSkipped files ({len(skipped)}):")
for f, reason in skipped:
    print(f"  {f} — {reason}")

Missing button_3_correct.numClicks in 083_explain2_2026-06-10_12h14.38.063.csv
Missing button_3_correct.numClicks in 083_explain2_2026-06-10_14h39.04.907.csv
Missing button_3_correct.numClicks in 135_explain2_2026-06-10_14h29.07.465.csv
Missing button_3_correct.numClicks in 181_explain2_2026-06-10_14h37.59.084.csv
Missing button_3_correct.numClicks in 252_explain2_2026-06-10_14h41.04.395.csv
Could not read PARTICIPANT_Accommodate5-5_2026-06-10_08h23.55.074.csv: No columns to parse from file
Could not read PARTICIPANT_Accommodate5-5_2026-06-10_13h30.01.886.csv: No columns to parse from file
Could not read PARTICIPANT_Accommodate5-5_2026-06-10_14h05.40.918.csv: No columns to parse from file
MOVED: 037_explain2_2026-06-10_11h33.17.761.csv
MOVED: 071_explain2_2026-06-10_12h39.40.808.csv
Missing button_end.numClicks in 083_explain2_2026-06-10_12h14.38.063.csv
MOVED: 083_explain2_2026-06-10_14h26.01.648.csv
Missing button_end.numClicks in 083_explain2_2026-06-10_14h39.04.907.csv
MOVED: 091_e